In [0]:
# dbutils.fs.rm(
#     "abfss://bronze@emedallionstore01.dfs.core.windows.net/schema/carts",
#     True
# )
# dbutils.fs.rm(
#     "abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/carts",
#     True
# )
# dbutils.fs.rm(
#     "abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/carts",
#     True
# )
# dbutils.fs.rm(
#     "abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/carts",
#     True
 # )
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/schema/products", True)
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/products", True)
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/products", True)
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/schema/users", True)
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/users", True)
# dbutils.fs.rm("abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/users", True)
# spark.sql(f"""DROP TABLE IF EXISTS emain_databricks.bronze.products""")
# spark.sql(f"""DROP TABLE IF EXISTS emain_databricks.bronze.users""")


DataFrame[]

In [0]:
from pyspark.sql.functions import col

SCHEMA_MAPPING = {

    "products": {
        "id": "int",
        "price": "double",
        "stock": "int",
        "rating": "double",
        "discountPercentage": "double",
        "weight": "double",
        "minimumOrderQuantity": "int"
    },

    "users": {
        "id": "int",
        "age": "int",
        "height": "double",
        "weight": "double"
    },

    "carts": {
        "id": "int",
        "userId": "int",
        "total": "double",
        "discountedTotal": "double",
        "totalProducts": "int",
        "totalQuantity": "int"
    }

}

In [0]:
from pyspark.sql.functions import col

def apply_schema(df, dataset):

    schema = SCHEMA_MAPPING[dataset]

    for column_name, datatype in schema.items():

        df = df.withColumn(
            column_name,
            col(column_name).cast(datatype)
        )

    return df

In [0]:
%python
df=(
    spark.readStream.format("cloudFiles").option("cloudFiles.format","json").option("cloudFiles.schemaLocation",schema_path).option("cloudFiles.inferColumnTypes", "true").load(source_path)

)

(
    df.writeStream.format("delta").option("checkpointLocation",checkpoint_path).outputMode("append").trigger(availableNow=True).start(delta_path).awaitTermination()
)

In [0]:
%python
df1=(
    spark.readStream.format("cloudFiles").option("cloudFiles.format","json").option("cloudFiles.inferColumnTypes", "true").option("cloudFiles.schemaLocation","abfss://bronze@emedallionstore01.dfs.core.windows.net/schema/users").load("abfss://bronze@emedallionstore01.dfs.core.windows.net/raw/users/")
)
(
    df1.writeStream.format("delta").option("checkpointLocation","abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/users").outputMode("append").trigger(availableNow=True).start("abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/users").awaitTermination()
)
df2=(
    spark.readStream.format("cloudFiles").option("cloudFiles.format","json").option("cloudFiles.inferColumnTypes", "true").option("cloudFiles.schemaLocation","abfss://bronze@emedallionstore01.dfs.core.windows.net/schema/carts").load("abfss://bronze@emedallionstore01.dfs.core.windows.net/raw/carts/")
)
(
    df2.writeStream.format("delta").option("checkpointLocation","abfss://bronze@emedallionstore01.dfs.core.windows.net/checkpoints/carts").outputMode("append").trigger(availableNow=True).start("abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/carts").awaitTermination()
)



In [0]:
%sql
DESCRIBE CATALOG EXTENDED emain_databricks;

info_name,info_value
Catalog Name,emain_databricks
Comment,
Owner,_workspace_admins_emain_databricks_7405618878476886
Catalog Type,Regular
Created By,@nk7756617gmail.onmicrosoft.com
Created At,2026-06-30 AD at 08:31:32 UTC
Updated By,@nk7756617gmail.onmicrosoft.com
Updated At,2026-06-30 AD at 08:31:35 UTC
Storage Root,abfss://unity-catalog-storage@dbstorageckts2hv6l34ps.dfs.core.windows.net/7405618878476886
Storage Location,abfss://unity-catalog-storage@dbstorageckts2hv6l34ps.dfs.core.windows.net/7405618878476886/__unitystorage/catalogs/e49c7b46-8b13-4c6f-9d07-4b89036a761f


In [0]:
%sql
create table if not exists emain_databricks.bronze.products 
using delta
location 'abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/products'

In [0]:
%sql
create table if not exists emain_databricks.bronze.users 
using delta
location 'abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/users'

In [0]:
%sql
create table if not exists emain_databricks.bronze.carts
using delta
location 'abfss://bronze@emedallionstore01.dfs.core.windows.net/delta/carts'

In [0]:
%sql
select count(*) from emain_databricks.bronze.products

count(*)
4074


In [0]:
%sql
select count(*)
from emain_databricks.bronze.products





count(*)
4074


In [0]:
%sql
select count(*)
from emain_databricks.bronze.carts

count(*)
4368


In [0]:
%sql
select count(*)
from emain_databricks.bronze.users

count(*)
4368


In [0]:
df.printSchema()

root
 |-- availabilityStatus: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- depth: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- width: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- id: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- qrCode: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |-- minimumOrderQuantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- reviews: array (nullable = true)
 |    |-- elemen

In [0]:
spark.read.format("delta").load(delta_path).printSchema()

root
 |-- availabilityStatus: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- depth: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- width: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- id: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- qrCode: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |-- minimumOrderQuantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- reviews: array (nullable = true)
 |    |-- elemen

In [0]:
spark.table("emain_databricks.bronze.products").printSchema()
spark.table("emain_databricks.bronze.users").printSchema()
spark.table("emain_databricks.bronze.carts").printSchema()

root
 |-- availabilityStatus: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- depth: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- width: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- id: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- qrCode: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |-- minimumOrderQuantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- reviews: array (nullable = true)
 |    |-- elemen